# Costco Analytics — Session 1: Meet the numbers

**Goal:** Use audited, real-world company data to explore revenue growth, margins, and inventory efficiency **before** choosing a fancy model.

**Source:** [Costco fiscal 2026 Form 10-K](https://www.sec.gov/Archives/edgar/data/909832/000090983226000093/cost-20260830.htm), consolidated income statements (fiscal 2024–26) and balance sheets (fiscal 2025–26). Figures are in **USD millions**, not dollars.

We deliberately start with an offline CSV to make the first calculations reproducible. Later cells can download Costco's large Company Facts JSON directly from SEC EDGAR. Nothing in the CSV is fabricated. Blank values mean **not included in the two-year balance-sheet comparison**, not zero.

## 0 — Setup

Install requirements from the project directory with `python -m pip install -r requirements.txt` (one time). You can run this notebook in JupyterLab, VS Code's notebook interface, or another local Python notebook environment.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Works whether the notebook is opened from the project root or notebooks/.
project = Path.cwd()
if not (project / "data" / "costco_2026_10k_sample.csv").exists():
    project = project.parent

sample_file = project / "data" / "costco_2026_10k_sample.csv"
df = pd.read_csv(sample_file).sort_values("fiscal_year").reset_index(drop=True)
print("Rows, columns:", df.shape)
display(df.drop(columns=["source_url"]))

## 1 — Check the accounting identities

Income statements give you fiscal-period **flows**; balance sheets give you period-end **stocks**. Compare net sales to total revenue: Costco also charges membership fees. Always check units.

In [ ]:
df["revenue_reconciliation_gap"] = (
    df["net_sales_usd_m"] + df["membership_fees_usd_m"] - df["total_revenue_usd_m"]
)
print("Revenue reconciliation errors (should all be 0):")
print(df[["fiscal_year", "revenue_reconciliation_gap"]].to_string(index=False))
assert (df["revenue_reconciliation_gap"] == 0).all()

## 2 — Create three metrics yourself

- **Sales growth:** (current net sales / prior-year net sales) − 1.
- **Net profit margin:** net income / total revenue. (Different revenue bases have different meanings.)
- **Operating margin:** operating income / total revenue.

We are not making a company investment recommendation from three rows.

In [ ]:
df["net_sales_growth_pct"] = df["net_sales_usd_m"].pct_change() * 100
df["net_profit_margin_pct"] = df["net_income_usd_m"] / df["total_revenue_usd_m"] * 100
df["operating_margin_pct"] = df["operating_income_usd_m"] / df["total_revenue_usd_m"] * 100

cols = ["fiscal_year", "net_sales_growth_pct", "net_profit_margin_pct", "operating_margin_pct"]
display(df[cols].round(2))

## 3 — Plot the sales trend

Build the pretty picture **after** understanding the variables.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(df["fiscal_year"].astype(str), df["net_sales_usd_m"], marker="o")
ax.set(title="Costco net sales", xlabel="Fiscal year", ylabel="USD millions")
ax.grid(alpha=0.25)
plt.show()

## 4 — Inventory and payables: where Supply Chain enters

2026 **Days Inventory Outstanding (DIO)** = 365 × average(2025, 2026 inventory) / 2026 merchandise costs. We also estimate **Days Payables Outstanding (DPO)** using merchandise costs as a **rough proxy** for purchases (actual supplier purchases would be preferable).

These are *aggregated company-level approximations*, not store-by-store measures. There is no 2024 balance-sheet amount in this small source extract, so a proper 2025 turnover calculation is unavailable here.

In [ ]:
inventory_2025 = float(df.loc[df.fiscal_year == 2025, "inventory_usd_m"].iloc[0])
inventory_2026 = float(df.loc[df.fiscal_year == 2026, "inventory_usd_m"].iloc[0])
payable_2025 = float(df.loc[df.fiscal_year == 2025, "accounts_payable_usd_m"].iloc[0])
payable_2026 = float(df.loc[df.fiscal_year == 2026, "accounts_payable_usd_m"].iloc[0])
merchandise_cost_2026 = float(df.loc[df.fiscal_year == 2026, "merchandise_costs_usd_m"].iloc[0])

avg_inventory = (inventory_2025 + inventory_2026) / 2
avg_payables = (payable_2025 + payable_2026) / 2
DIO_2026 = 365 * avg_inventory / merchandise_cost_2026
DPO_2026_proxy = 365 * avg_payables / merchandise_cost_2026
print(f"2026 DIO: {DIO_2026:.2f} days")
print(f"2026 DPO (COGS proxy, not true purchases): {DPO_2026_proxy:.2f} days")

## 5 — Save your first analysis file

In [ ]:
out = project / "data" / "costco_first_analysis.csv"
df.to_csv(out, index=False)
print("Created:", out)

## 6 — OPTIONAL: download Costco's entire SEC Company Facts JSON

This is **not required** for the exercises above. It requires an internet connection; the SEC might reject generic or misidentified automated requests. Put your **real contact email** in the User-Agent string and run this cell once. The SEC's Company Facts API includes tagged values across years and filings; later we'll clean those into a fuller time series. An SEC filing contains repeated historical facts, so `fiscal_year` and filing year are **not** always the same observation period; inspect `start`, `end`, `filed` and `form` before calculations.

SEC source: https://www.sec.gov/search-filings/edgar-application-programming-interfaces

In [ ]:
import requests

YOUR_EMAIL = "REPLACE_WITH_YOUR_REAL_EMAIL@example.com"
if YOUR_EMAIL.startswith("REPLACE_"):
    print("Edit YOUR_EMAIL first; offline exercise already works.")
else:
    endpoint = "https://data.sec.gov/api/xbrl/companyfacts/CIK0000909832.json"
    response = requests.get(
        endpoint,
        headers={"User-Agent": f"Independent analytics learning {YOUR_EMAIL}"},
        timeout=40,
    )
    response.raise_for_status()
    costco_facts = response.json()
    raw_file = project / "data" / "costco_companyfacts_full.json"
    raw_file.write_text(response.text, encoding="utf-8")
    print("Saved live SEC JSON:", raw_file, "with", len(costco_facts.get("facts",{}).get("us-gaap",{})), "us-gaap concepts")

## 7 — OPTIONAL: search the financial tags (run after successful Section 6 download)

In [ ]:
import json
raw_file = project / "data" / "costco_companyfacts_full.json"
if raw_file.exists():
    costco_facts = json.loads(raw_file.read_text(encoding="utf-8"))
    tag_list = []
    for taxonomy, concepts in costco_facts.get("facts", {}).items():
        for tag, entry in concepts.items():
            tag_list.append({"taxonomy": taxonomy, "tag": tag, "label": entry.get("label", "")})
    tags = pd.DataFrame(tag_list)
    display(tags[tags["tag"].str.contains("inventory|inventories|payable|revenue|sales|costof", case=False, regex=True)].head(70))
else:
    print("Download the SEC JSON in Section 6 first. The offline exercises are complete.")

## Think like an analyst

Before adding Walmart or building an NPV model:

1. Why can revenue grow while profit margin falls?
2. Is lower DIO *always* better? What about stockouts?
3. Costco states it sometimes sells merchandise before having to pay suppliers. How might that relate to your DIO and DPO results?
4. What additional years or variables would we need to study persistent inventory efficiency?

**Important:** NPV requires *incremental expected project cash flows* and a defensible discount rate. A 10-K's historical revenue is **not** automatically an NPV project cash-flow forecast.